# Path and parameter settings

In [18]:
import os
import glob
import numpy as np
import rasterio
import zarr
import numcodecs
from collections import defaultdict
from datetime import datetime
import shutil

year = 2019   # The predicted year
interval = 16 # The time interval for reconstructed dataset
freq = 'md' if year >= 2000 else 'av'# Select the coarse-resolution dataset based on the year.

# A.NDVI source path
# including collection_mdpr、collection_ltpr、collection_mdre、collection_ltre、collection_avpr、collection_avre
base = r'E:\data\HANTS\python_exp\test'    # ALL datasets are stored below the "base" directory.
mdpr_path = fr'{base}\collection_{freq}pr' # MODIS(or AVHRR) NDVI of predicted year
ltpr_path = fr'{base}\collection_ltpr'     # Landsat NDVI of predicted year
mdre_path = fr'{base}\collection_{freq}re' # MODIS(or AVHRR) NDVI of reference year
ltre_path = fr'{base}\collection_ltre'     # Landsat NDVI of reference year

# B.Customize the storage path for Zarr arrays
# B.a Transform NDVI to zarr arrays
ltpr_lists_path   = fr"{base}\ltpr_lists.zarr" # Landsat NDVI of predicted year
mdpr_lists_path   = fr"{base}\mdpr_lists.zarr" # MODIS NDVI of predicted year
mdre_lists_path   = fr'{base}\mdre_lists.zarr' # MODIS NDVI of reference year
ltre_lists_path   = fr'{base}\ltre_lists.zarr' # Landsat NDVI of reference year
# B.b Transform harmonic coefficient (coefs) to zarr arrays
coefImg_ltpr_path = fr"{base}\coefImg_ltpr_.zarr" # Landsat coefs of predicted year
coefImg_mdpr_path = fr"{base}\coefImg_mdpr_.zarr" # MODIS coefs of predicted year
coefImg_mdre_path = fr'{base}\coefImg_mdre_.zarr' # MODIS coefs of reference year
coefImg_ltre_path = fr'{base}\coefImg_ltre_.zarr' # Landsat coefs of reference year
# B.c Transform other file to zarr arrays
output_dir = fr'{base}\FMNTS' # output NDVI of predicted year
temp_zarr_dir_pr  = fr"{base}\temp_pr"   # Temporary file of predicted year
temp_zarr_dir_re  = fr'{base}\temp_re'   # Temporary file of reference year
hants_lists_path  = ltpr_lists_path
coefImg_hants_path= fr"{base}\coefImg_hants_.zarr"
weight_zarr_path  = fr"{base}\weight_.zarr"    # weight of predicted year
# ============ Configuration parameters ============
# C. Dask parameters
H_chunk = 64
W_chunk = 64
n_workers = 4
threads_per_worker = 1
memory_limit_per_worker = '6GB'

# Load target year data

In [2]:
# ============ Function 1: Read images and group them (only record metadata (path, date, shape, projection)). ============
def collect_landsat_metadata(ltpr_path):
    """Collect metadata from Landsat files and group them by date"""
    ltpr_files = glob.glob(os.path.join(ltpr_path, '*.tif'))
    dates_groups = defaultdict(list)
    
    # Get image dimensions (from the first file)
    with rasterio.open(ltpr_files[0]) as src:
        H, W = src.shape
        transform = src.transform
    
    for tif in ltpr_files:
        filename = os.path.basename(tif)
        date_str = filename.split('_')[-1].split('.')[0]
        dateday = datetime.strptime(date_str, '%Y%m%d').date()
        dates_groups[dateday].append(tif)  # # dates_groups only stores file paths
    
    print(f"Found {len(ltpr_files)} files, grouped into {len(dates_groups)} dates")
    return dates_groups, H, W, transform

# ============ Function 2: Create a temporary Zarr to store the original image. ============
def create_temp_zarr(dates_groups, H, W, temp_zarr_path):
    '''
    temp_zarr_path contains a Zarr root group (container) root, which stores multiple Zarr arrays (n_imgs, H, W), each Zarr array corresponding to a date.
    temp_zarr_path/       # Zarr's root directory
        ├── .zgroup       # Zarr's metadata file
        ├── 2023-01-01    # The Zarr dataset stores all images from January 1, 2023.
        ├── 2023-01-02    # 
        ├── 2023-01-03    # 
        └── ...           
    '''
    
    if os.path.exists(temp_zarr_path):# Check if the temporary Zarr directory exists; if it does, force delete the directory and all its contents.
        shutil.rmtree(temp_zarr_path)
    
    store = zarr.DirectoryStore(temp_zarr_path) # Initialize Zarr storage directory (folder)
    root = zarr.group(store=store)              # Create a Zarr root group
    compressor = numcodecs.Blosc(cname='zstd', clevel=3)# Create a compressor to compress the Zarr array
    
    date_to_idx = {}  # Date to index mapping
    sorted_dates = sorted(dates_groups.keys())  # Get the sorted dates
    
    for idx, date_key in enumerate(sorted_dates): #idx is the index of each date, and date_key is each date; dates will not be repeated.
        file_list = dates_groups[date_key]        # Path to all images for the current date
        n_imgs = len(file_list)
        
        # Create a dataset (n_imgs, H, W) for this date and store it in the root group.
        ds = root.create_dataset(
            str(date_key),       # Convert date to string format
            shape=(n_imgs, H, W), 
            chunks=(1, min(512, H), min(512, W)), # Define the chunk size of the dataset.
            dtype='f4',  #float32
            compressor=compressor # Compress data using the zstd algorithm, compression level 3
        )
        
        # Read and write
        for i, tif_path in enumerate(file_list):# Path to all images for the current date
            with rasterio.open(tif_path) as src:
                ndvi = src.read(1).astype('f4')
                # Apply a mask
                ndvi = np.where((ndvi >= -0.5) & (ndvi <= 1), ndvi, np.nan)
                ds[i, :, :] = ndvi # ndvi is stored in the dataset ds(n_imgs, H, W).
        
        date_to_idx[date_key] = idx#One date corresponds to one index, which in turn corresponds to one ds dataset.
    
    return sorted_dates, date_to_idx# Returns the date and its corresponding index

# ============ Function 3: Using Zarr for maximum value synthesis ============
def max_composite_zarr(temp_zarr_path, sorted_dates, output_zarr_path):
    if os.path.exists(output_zarr_path):# Check if the temporary Zarr directory exists; if it does, force delete the directory and all its contents.
        shutil.rmtree(output_zarr_path)
    
    temp_root = zarr.open(temp_zarr_path, mode='r') # Open the temporary Zarr root group in read mode
    
    # Get Size
    first_ds = temp_root[str(sorted_dates[0])] # Get the Zarr dataset for the first date
    H, W = first_ds.shape[1], first_ds.shape[2]# Get the image's height H and width W
    T = len(sorted_dates)                      # Get the time dimension T, which is the total number of dates.
    
    # Create output Zarr
    out_store = zarr.DirectoryStore(output_zarr_path)  # Initialize a new Zarr store
    out_root = zarr.group(store=out_store)             # Create root group
    compressor = numcodecs.Blosc(cname='zstd', clevel=3, shuffle=numcodecs.Blosc.SHUFFLE)# Compress Zarr storage
    
    # Create a dataset (T, H, W) ltpr_lists containing each date
    out_ds = out_root.create_dataset(
        'ltpr_lists',
        shape=(T, H, W),
        chunks=(T, 64, 64),  # spatial segmentation
        dtype='f4',
        compressor=compressor
    )
    
    # Combine maximum values for each date
    for t, date_key in enumerate(sorted_dates):
        ds = temp_root[str(date_key)]  # (n_imgs, H, W)
        # Use Zarr's nanomax (if n_imgs is large, it can be processed in blocks)
        max_composite = np.nanmax(ds[:], axis=0)  # (H, W)
        out_ds[t, :, :] = max_composite    # The data after maximum value synthesis is stored in the dataset out_ds
    
    return sorted_dates

# ============ Function 4: Process MODIS data (write directly to Zarr) ============
def process_modis_to_zarr(mdpr_folder, output_zarr_path):
    """Read MODIS files and write them directly to Zarr"""
    import shutil
    if os.path.exists(output_zarr_path):
        shutil.rmtree(output_zarr_path)
    
    # Collect file and date
    files = sorted(os.listdir(mdpr_folder))
    dates = []
    for file in files:
        date_str = file.replace('.tif', '')
        dateday = datetime.strptime(date_str, '%Y_%m_%d').date()
        dates.append(dateday)
    
    # Read the first file and get its dimensions
    first_path = os.path.join(mdpr_folder, files[0])
    with rasterio.open(first_path) as src:
        H, W = src.shape
    
    T = len(files)
    
    # Create Zarr
    store = zarr.DirectoryStore(output_zarr_path)
    root = zarr.group(store=store)
    compressor = numcodecs.Blosc(cname='zstd', clevel=3, shuffle=numcodecs.Blosc.SHUFFLE)
    
    ds = root.create_dataset(
        'mdpr_lists',
        shape=(T, H, W),
        chunks=(T, 64, 64),
        dtype='f4',
        compressor=compressor
    )
    
    print(f"Writing {T} MODIS frames to zarr...")
    
    for t, (file, date) in enumerate(zip(files, dates)):
        file_path = os.path.join(mdpr_folder, file)
        with rasterio.open(file_path) as src:
            ndvi = src.read(1).astype('f4')
        ds[t, :, :] = ndvi
    
    return np.array(dates)

# ============ Main step ============
if __name__ == "__main__":

    print("Step 1: Processing Landsat data")
    
    # 1. Collect Landsat metadata
    dates_groups, H, W, transform = collect_landsat_metadata(ltpr_path)
    
    # 2. Create temporary Zarr
    temp_zarr_path = os.path.join(temp_zarr_dir_pr, f"landsat_{year}_raw.zarr")
    sorted_dates, date_to_idx = create_temp_zarr(dates_groups, H, W, temp_zarr_path)
    
    # 3. The maximum value is synthesized and written into the final Zarr
    ltpr_dates = max_composite_zarr(temp_zarr_path, sorted_dates, ltpr_lists_path)
    ltpr_dates = np.array(sorted_dates)
    
    print("Step 2: Processing MODIS data")
    
    # 4. Process MODIS Data
    mdpr_dates = process_modis_to_zarr(mdpr_path, mdpr_lists_path)
    
    print(f"Landsat zarr: {ltpr_lists_path}")
    print(f"  - Shape: {zarr.open(ltpr_lists_path)['ltpr_lists'].shape}")
    print(f"  - Dates: {len(ltpr_dates)}")
    print(f"MODIS zarr: {mdpr_lists_path}")
    print(f"  - Shape: {zarr.open(mdpr_lists_path)['mdpr_lists'].shape}")
    print(f"  - Dates: {len(mdpr_dates)}")


Step 1: Processing Landsat data
Found 36 files, grouped into 36 dates


C:\Users\fengcanzu\AppData\Local\Temp\ipykernel_37924\2722733862.py:98: RuntimeWarning: All-NaN slice encountered
  max_composite = np.nanmax(ds[:], axis=0)  # (H, W)


Step 2: Processing MODIS data
Writing 46 MODIS frames to zarr...
Landsat zarr: E:\data\HANTS\python_exp\test\ltpr_lists.zarr
  - Shape: (36, 30, 38)
  - Dates: 36
MODIS zarr: E:\data\HANTS\python_exp\test\mdpr_lists.zarr
  - Shape: (46, 30, 38)
  - Dates: 46


# Reading data from multiple years to synthesize reference sequences

In [3]:
import os
import gc
import glob
import shutil
import numpy as np
import rasterio
import zarr
import numcodecs
from collections import defaultdict
from datetime import datetime, timedelta, date
import dask.array as da
from dask.distributed import Client

# ============ Reference sequence date ============
current_date = date(2020, 1, 1)
eight_pairs = []
while current_date <= date(2020, 12, 31):
    eight_pairs.append(current_date)
    current_date += timedelta(days=8)

# ============ Function 1: Collect file metadata and group by date ============
def collect_files_by_date(file_pattern, date_format='%Y_%m_%d', date_extract_func=None):
    """
    Collect the documents and group them by date (change to 2020).
    
    Args:
        file_pattern: glob mode
        date_format: Date string format
        date_extract_func: Custom date extraction function (filename) -> date_str
    """
    files = glob.glob(file_pattern)
    date_dict = defaultdict(list)
    
    for tif in files:
        filename = os.path.basename(tif)
        
        # Extract date string
        if date_extract_func:
            date_str = date_extract_func(filename)
        else:
            date_str = filename.replace('.tif', '')
        
        # Parse the date and change it to 2020.
        dateday = datetime.strptime(date_str, date_format).date()
        new_date = dateday.replace(year=2020)
        
        date_dict[new_date].append(tif)
    
    sorted_dates = sorted(date_dict.keys())
    print(f"Found {len(files)} files, grouped into {len(sorted_dates)} dates")
    
    return date_dict, sorted_dates, files

# ============ Function 2: Create a temporary Zarr store for the original image (by date) ============
def create_temp_zarr_by_date(date_dict, sorted_dates, temp_zarr_path, value_range=(-0.2, 1)):
    """
    Store the image for each date in a temporary Zarr and apply a mask.
    """
    if os.path.exists(temp_zarr_path):
        shutil.rmtree(temp_zarr_path)
    
    # Read the first file to get the size
    first_file = date_dict[sorted_dates[0]][0]
    with rasterio.open(first_file) as src:
        H, W = src.shape
    
    store = zarr.DirectoryStore(temp_zarr_path)
    root = zarr.group(store=store)
    compressor = numcodecs.Blosc(cname='zstd', clevel=3)
    
    print(f"Creating temp zarr: {temp_zarr_path}")
    
    for idx, date_key in enumerate(sorted_dates):
        file_list = date_dict[date_key]
        n_imgs = len(file_list)
        
        # Create a dataset (n_imgs, H, W) for this date.
        ds = root.create_dataset(
            str(date_key),
            shape=(n_imgs, H, W),
            chunks=(1, min(512, H), min(512, W)),
            dtype='f4',
            compressor=compressor
        )
        
        # Read and write
        for i, tif_path in enumerate(file_list):
            with rasterio.open(tif_path) as src:
                ndvi = src.read(1).astype('f4')
                # Application of masks
                ndvi = np.where((ndvi >= value_range[0]) & (ndvi <= value_range[1]), ndvi, np.nan)
                ds[i, :, :] = ndvi
        
        if (idx + 1) % 10 == 0 or idx == len(sorted_dates) - 1:
            print(f"  Stored {idx + 1}/{len(sorted_dates)} dates")
    
    return H, W

# ============ Function 3: 8-day window maximum value synthesis (using Zarr) ============
def apply_landsat_mask_zarr_parallel(zarr_path, dataset_name, chunk_size=(1, 512, 512), n_workers=4):
    # 1. Initialize Dask cluster
    with Client(n_workers=n_workers, threads_per_worker=1) as client:
        print(f"Dask Cluster Startup | Dashboard: {client.dashboard_link}")
 
        # 2. Safe loading of Zarr arrays
        with zarr.open(zarr_path, mode='r+') as root:
            ds = root[dataset_name]
            print(f"Original blocks: {ds.chunks} | Target block: {chunk_size}")
 
            # 3. Dynamically adjust the block partitioning strategy
            dask_arr = da.from_zarr(
                zarr_path,
                component=dataset_name,
                chunks=chunk_size if chunk_size else ds.chunks
            ).rechunk(chunk_size)
 
            # 4. Block-based calculation of statistics (memory-optimized version)
            with np.errstate(invalid='ignore'):
                max_val = da.nanmax(dask_arr, axis=0)
                mean_val = da.nanmean(dask_arr, axis=0)
                mask = (max_val < 0.2) | (mean_val < 0.1) # Vegetation mask
 
            # 5. Write-back in chunks (streaming processing)
            masked = da.where(mask[None, :, :], np.nan, dask_arr) # None adds a dimension, expanding the two-dimensional mask into a three-dimensional one.
            
            _ = masked.to_zarr(
                zarr_path,
                component=dataset_name,
                overwrite=True,
                compute=True,  # Explicitly triggered calculation
                return_stored=False
            )
 
    print(f"✅ Parallel masking completed | Blocking: {chunk_size} | Work process: {n_workers}")

def max_composite_8day_zarr_dask(temp_zarr_path, sorted_dates, eight_pairs, output_zarr_path, dataset_name):
    """Use Dayk to divide the process into blocks in the spatial dimension"""    
    if os.path.exists(output_zarr_path):
        shutil.rmtree(output_zarr_path)
    
    temp_root = zarr.open(temp_zarr_path, mode='r')
    first_ds = temp_root[str(sorted_dates[0])]
    H, W = first_ds.shape[1], first_ds.shape[2]
    T = len(eight_pairs)
   
    out_store = zarr.DirectoryStore(output_zarr_path)
    out_root = zarr.group(store=out_store)
    compressor = numcodecs.Blosc(cname='zstd', clevel=3, shuffle=numcodecs.Blosc.SHUFFLE)
    
    out_ds = out_root.create_dataset(
        dataset_name,
        shape=(T, H, W),
        chunks=(T, 64, 64),
        dtype='f4',
        compressor=compressor
    )
    
    print(f"Performing 8-day max composite with Dask...")
    result_dates = []
    
    for t, start_date in enumerate(eight_pairs):
        end_date = start_date + timedelta(days=7)
        
        # # Collect all Dask arrays within the window (without loading them into memory)
        dask_arrays = []
        
        for d in sorted_dates:
            if start_date <= d <= end_date:
                ds = temp_root[str(d)]
                for i in range(ds.shape[0]):
                    # ✅ Create a Dask array and read it in blocks.
                    arr = da.from_array(ds[i, :, :], chunks=(512, 512))
                    dask_arrays.append(arr)
        
        if len(dask_arrays) > 0:
            # ✅ Stack the data into a Dask array and perform block computation.
            stacked = da.stack(dask_arrays, axis=0)  
            max_composite = da.nanmax(stacked, axis=0)  
            
            # ✅ Compute in chunks and write.
            result = max_composite.compute()  # Load results only here
        else:
            result = np.full((H, W), np.nan, dtype='f4')
        
        out_ds[t, :, :] = result
        result_dates.append(start_date)
        
        if (t + 1) % 10 == 0 or t == T - 1:
            print(f"  Composited {t + 1}/{T} windows")
    
    return np.array(result_dates), H, W


def extract_landsat_date(filename):
    """Extract date from Landsat filename"""
    return filename.split('_')[-1].split('.')[0]

os.makedirs(temp_zarr_dir_re, exist_ok=True)
overwrite_output = False    
cleanup_temp=True           # Default cleaning of temporary files

## Synthesize MODIS reference sequences

In [4]:
if __name__ == "__main__":    
    # ============ MODIS processing ============
    print("=" * 70)
    print("Processing MODIS Reference Sequence")
    print("=" * 70)
        
    if not os.path.exists(mdre_lists_path) or overwrite_output:
        try:
            mdre_pattern = fr'{mdre_path}\*.tif'
            mdre_date_dict, mdre_sorted_dates, _ = collect_files_by_date(
                mdre_pattern, date_format='%Y_%m_%d' # MODIS date format
            )

            temp_mdre_path = os.path.join(temp_zarr_dir_re, 'mdre_raw.zarr')
            H, W = create_temp_zarr_by_date(mdre_date_dict, mdre_sorted_dates, temp_mdre_path)
            mdre_dates, _, _ = max_composite_8day_zarr_dask(
                temp_mdre_path, mdre_sorted_dates, eight_pairs, mdre_lists_path, 'mdre_lists'
            )
            
            # === clean memory ===
            del mdre_date_dict, mdre_sorted_dates
            if 'temp_root' in locals():  # clean zarr
                temp_root.store.close()  # Explicitly disable storage
                del temp_root 
            if os.path.exists(temp_mdre_path):  # clean teporary file
                shutil.rmtree(temp_mdre_path, ignore_errors=True)
            gc.collect()  
            
        except Exception as e:
            print(f"MODIS processing failed.: {str(e)}")
            raise 
    else:
        print(f"{mdre_lists_path} exists and overwrite_output=False -> skipping MODIS processing")
 
    print(f"MODIS zarr: {mdre_lists_path}")
    print(f"  - Shape: {zarr.open(mdre_lists_path)['mdre_lists'].shape}")

Processing MODIS Reference Sequence
Found 276 files, grouped into 84 dates
Creating temp zarr: E:\data\HANTS\python_exp\test\temp_re\mdre_raw.zarr
  Stored 10/84 dates
  Stored 20/84 dates
  Stored 30/84 dates
  Stored 40/84 dates
  Stored 50/84 dates
  Stored 60/84 dates
  Stored 70/84 dates
  Stored 80/84 dates
  Stored 84/84 dates
Performing 8-day max composite with Dask...
  Composited 10/46 windows
  Composited 20/46 windows
  Composited 30/46 windows
  Composited 40/46 windows
  Composited 46/46 windows
MODIS zarr: E:\data\HANTS\python_exp\test\mdre_lists.zarr
  - Shape: (46, 30, 38)


## Synthesize Landsat reference sequences

In [5]:
if __name__ == "__main__":
        # ============ Landsat process ============
    print("\n" + "=" * 70)
    print("Processing Landsat Reference Sequence")
    print("=" * 70)
    
    if not os.path.exists(ltre_lists_path) or overwrite_output:
        # 1. Processing Landsat reference sequences
        ltre_pattern = fr'{ltre_path}\*.tif'
           # Collect the documents and group them by date (change to 2020).
        ltre_date_dict, ltre_sorted_dates, _ = collect_files_by_date(
            ltre_pattern, date_format='%Y%m%d', date_extract_func=extract_landsat_date 
        )
        

        temp_ltre_path = os.path.join(temp_zarr_dir_re, 'ltre_raw.zarr')
        H, W = create_temp_zarr_by_date(ltre_date_dict, ltre_sorted_dates, temp_ltre_path)
        
        ltre_dates, _, _ = max_composite_8day_zarr_dask(
            temp_ltre_path, ltre_sorted_dates, eight_pairs, ltre_lists_path, 'ltre_lists'
        )
               
        # 2. Landsat mask 
        apply_landsat_mask_zarr_parallel(
            zarr_path=ltre_lists_path, dataset_name='ltre_lists',
            chunk_size=(1, 512, 512),   # Matches integer multiples of the original block (1,256,256).
            n_workers=min(8, os.cpu_count()//2)  # Recommend half the number of CPU cores
        )
    else:
        print(f"{ltre_lists_path} exists and overwrite_output=False -> skipping Landsat processing")
    
    # ============ Results Output and Cleanup ============
    print(f"Landsat zarr: {ltre_lists_path}")
    print(f"  - Shape: {zarr.open(ltre_lists_path)['ltre_lists'].shape}")
    
    # 3. Clean up temporary files (optional)
    if cleanup_temp:
        print("\nCleaning up temporary files...")
        shutil.rmtree(temp_zarr_dir_re)


Processing Landsat Reference Sequence
Found 120 files, grouped into 120 dates
Creating temp zarr: E:\data\HANTS\python_exp\test\temp_re\ltre_raw.zarr
  Stored 10/120 dates
  Stored 20/120 dates
  Stored 30/120 dates
  Stored 40/120 dates
  Stored 50/120 dates
  Stored 60/120 dates
  Stored 70/120 dates
  Stored 80/120 dates
  Stored 90/120 dates
  Stored 100/120 dates
  Stored 110/120 dates
  Stored 120/120 dates
Performing 8-day max composite with Dask...
  Composited 10/46 windows
  Composited 20/46 windows
  Composited 30/46 windows
  Composited 40/46 windows
  Composited 46/46 windows
Dask Cluster Startup | Dashboard: http://127.0.0.1:8787/status
Original blocks: (46, 64, 64) | Target block: (1, 512, 512)
✅ Parallel masking completed | Blocking: (1, 512, 512) | Work process: 6
Landsat zarr: E:\data\HANTS\python_exp\test\ltre_lists.zarr
  - Shape: (46, 30, 38)

Cleaning up temporary files...


# Define the harmonic fitting function

In [6]:
import numpy as np
from sklearn.linear_model import Ridge
import os
import shutil
import zarr
import numcodecs
import numpy as np
from datetime import datetime
from dask.distributed import Client, LocalCluster, as_completed
import multiprocessing

def hants_pixel(ndvi_values, dates, num_harm,iffitted):
    # Convert all dates to day offset (based on a certain base point, such as 2020-01-01)
    base_date = datetime(2020, 1, 1).date()
    
    def addDependents(dates):
        days = np.array([(d - base_date).days for d in dates])
        t = days / 365.25 * 2 * np.pi
        return t
    
    t_fit = addDependents(dates) #(n_sample,)  one-dimensional vector  
    

    # Construct the Fourier basis function matrix (excluding constant) → (n_sample, 2*num_harm)
    def addHarmonics(t, num_harm):
        components = [np.ones_like(t)]  
        for i in range(1, num_harm + 1):
            components.append(np.cos(i * t))
            components.append(np.sin(i * t))
        return np.stack(components, axis=1) # Shape: (n, 2*num_harm + 1)

    X = addHarmonics(t_fit, num_harm)       # Shape: (n_sample, 2*num_harm + 1)
    y = ndvi_values
    
    if len(y) < 2 * num_harm + 2:
        return np.full((2 * num_harm + 1,), np.nan)
        
    else:
        # 1. Establish a Ridge regression model
        ridge_model = Ridge(alpha=0.5, fit_intercept=False) # The intercept is not fitted because the harmonic term has 1.
        ridge_model.fit(X, y)

        # 2. Extract coefficients: Extract the coefficients from the fitting process.
        coefs = ridge_model.coef_        # cosntant，cos_1, sin_1, cos_2, sin_2, ...     shape:（2*num_harm + 1，)
        constant = coefs[0]
        
        #3.Constructing the harmonic coefficient matrix
        harmonic = [constant]
        for i in range(num_harm):
            cos_coef = coefs[2*i+1]
            sin_coef = coefs[2*i+2]
            amp = np.sqrt(cos_coef**2 + sin_coef**2)
            phase_angle = np.arctan2(sin_coef, cos_coef) * 180 / np.pi
            harmonic.extend([amp, phase_angle])#Add amplitude and phase，[con amp1 pha1 amp2 pha2...]

        ndvi_time = []
        fet = 0.05
        
        if iffitted==11: # First iteration
            y_pred = ridge_model.predict(X)
            mask02 = ((y_pred - y) < fet) 
            if len(dates[mask02]) < 2 * num_harm + 2:
                return hants_pixel(y, dates, num_harm, None)
            else:
                return hants_pixel(y[mask02], dates[mask02], num_harm, 12)
        
        elif iffitted==12: # Second iteration
            y_pred = ridge_model.predict(X)
            mask02 = (y_pred - y) < fet
            if len(dates[mask02]) < 2 * num_harm + 2:
                return hants_pixel(y, dates, num_harm, None)
            else:
                return hants_pixel(y[mask02], dates[mask02], num_harm, 13)

        elif iffitted==13: # Third iteration
            y_pred = ridge_model.predict(X)
            mask02 = (y_pred - y) < fet
            
            if len(dates[mask02]) < 2 * num_harm + 2:
                return hants_pixel(y, dates, num_harm, None)
            else:
                return hants_pixel(y[mask02], dates[mask02], num_harm, None)# Output the harmonic coefficients after three iterations        
      
        else:         # Iffitted is empty, return the constant, amplitude, and phase.
            return harmonic    #[con amp1 pha1 amp2 pha2...]

def worker_compute_block(dataset_zarr_path, dataset_name, h0, h1, w0, w1, dates_ts_local, nharm, iffitted, ncoef):
    """
    Read data for a spatial block from the specified Zarr file 
    and call the process_block_T_hw function to calculate the harmonic coefficients of the block.
    Opens dataset_zarr_path, reads dataset_name[:, h0:h1, w0:w1], returns (ncoef, h, w)
    """
    import zarr as _zarr # Independent initialization environment
    import numpy as _np
    zr = _zarr.open(dataset_zarr_path, mode='r')
    arr = zr[dataset_name][:, h0:h1, w0:w1]   # (T, hsize, wsize)
    arr = _np.asarray(arr, dtype=_np.float32)
    res = process_block_T_hw(arr, dates_ts_local, nharm, iffitted, ncoef) # ncoef：1 + 2 * nharm
    return res

def process_block_T_hw(block_t_hw, dates_ts_local, nharm, iffitted, ncoef):# Calculate the harmonic coefficients for the block
    T, bh, bw = block_t_hw.shape
    out = np.full((ncoef, bh, bw), np.nan, dtype=np.float32)
    for ii in range(bh):
        for jj in range(bw):
            series = block_t_hw[:, ii, jj]
            valid = ~np.isnan(series)
            if valid.sum() == 0:
                continue
            vals = series[valid]              #Valid data
            dates_sel = dates_ts_local[valid] #Valid date
            try:
                coef = hants_pixel(vals, dates_sel, nharm, iffitted) #  [con amp1 pha1 amp2 pha2...]
                coef = np.asarray(coef, dtype=np.float32)    # transfer to numpy  float32
                out[:, ii, jj] = coef 
            except Exception:
                out[:, ii, jj] = np.nan
    return out

# --- driver to run worker_submit for one dataset ---
def run_for_dataset(dataset_zarr_path, dataset_name, dataset_dates, out_zarr_path,
                    nharm=3, iffitted=2, overwrite_output=True):
    """
    dataset_zarr_path: Existing input zarr path
    dataset_name: Dataset Name (e.g. 'ltpr_lists' or 'mdpr_lists')
    dataset_dates: Date array
    out_zarr_path: Output harmonic coefficient zarr path
    """
    ncoef = 1 + 2 * nharm

    # 1) Read the existing zarr directly， ltpr_lists_path、mdpr_lists_path
    zr = zarr.open(dataset_zarr_path, mode='r')
    in_ds = zr[dataset_name]  #zr['ltpr_lists']、zr['mdpr_lists']
    T, H, W = in_ds.shape

    # 2) prepare output zarr    coefImg_ltpr_path、coefImg_mdpr_path
    if os.path.exists(out_zarr_path) and overwrite_output:
        shutil.rmtree(out_zarr_path)
    out_store = zarr.DirectoryStore(out_zarr_path)         #初始化存储
    out_root = zarr.group(store=out_store, overwrite=True) #Create the root group out_root
    compressor = numcodecs.Blosc(cname='zstd', clevel=3, shuffle=numcodecs.Blosc.SHUFFLE)
    # Create the zarr harmonic coefficients out_ds in the root group out_root.
    out_ds = out_root.create_dataset('coefs', shape=(ncoef, H, W), chunks=(ncoef, H_chunk, W_chunk), dtype='f4', compressor=compressor)
    
    
    #3) The spatial chunk plan calculates a suitable chunking scheme based on the chunk size of the input list array and the chunk size of the output coef array.
    raw_h_chunks = in_ds.chunks[1] # zarr数组会存有分块信息 height
    raw_w_chunks = in_ds.chunks[2] # weight
    
    def normalize_chunks(raw, full): 
        if isinstance(raw, int):
            size = raw            # Height of the list array (number of rows)
            n_full = full // size # The coef height divided by the lists height, where the divisor is the number of completely filled blocks.
            rem = full % size     # Remainder, the remaining columns or rows
            parts = [size] * n_full#Create a list of n_full blocks of the same size.
            if rem: parts.append(rem)
                
            return tuple(parts)
        else:
            return tuple(raw)
    
    h_chunks = normalize_chunks(raw_h_chunks, H)
    w_chunks = normalize_chunks(raw_w_chunks, W)
    '''
    170 / 64 = 2 余 42 h_chunks[64,64,42]       The 170 lines were divided into 64, 64, and 42 lines.
    197 / 64 = 3 余 5  w_chunks[64, 64, 64, 5]  197 columns were divided into 64, 64, 64, and 5 columns.
    '''   
    print("Spatial chunking H:", h_chunks, "W:", w_chunks)

    # 4) start dask cluster
    cluster = LocalCluster(n_workers=n_workers, threads_per_worker=threads_per_worker, memory_limit=memory_limit_per_worker)
    client = Client(cluster)
    print("Dask client:", client)

    # 5) submit tasks
    dates_ts = dataset_dates
    future_map = {}
    for hi, hsize in enumerate(h_chunks):
        h0 = sum(h_chunks[:hi])
        h1 = h0 + hsize
        for wi, wsize in enumerate(w_chunks):
            w0 = sum(w_chunks[:wi])
            w1 = w0 + wsize
            f = client.submit(worker_compute_block, dataset_zarr_path, dataset_name, h0, h1, w0, w1,
                              dates_ts, nharm, iffitted, ncoef, pure=False)
            future_map[f] = (h0, h1, w0, w1)
    total = len(future_map)
    print(f"Submitted {total} tasks for dataset {dataset_name}.")

    # 6) collect results and write
    ac = as_completed(list(future_map.keys()))
    done = 0
    for future in ac:
        try:
            res = future.result()
        except Exception as e:
            print("Worker failed:", e)
            continue
        h0, h1, w0, w1 = future_map[future]
        out_ds[:, h0:h1, w0:w1] = res.astype('f4')
        done += 1
        if done % 10 == 0 or done == total:
            print(f"Wrote {done}/{total} blocks for {dataset_name} (last block h[{h0}:{h1}] w[{w0}:{w1}])")

    client.close()
    cluster.close()
    print(f"Finished dataset {dataset_name}. Output at: {out_zarr_path}")



# Parallel processing

## Calculate the harmonic coefficients of the reference image (calculate only once).

In [7]:
# ------------------ Run pipeline for all datasets ------------------
if __name__ == "__main__":
    datasets = [ 
        # (Dataset name, zarr path, date array, harmonic number, output zarr path, iffitted parameter)
        ('ltre_lists', ltre_lists_path, ltre_dates, 3, coefImg_ltre_path, 2),
        ('mdre_lists', mdre_lists_path, mdre_dates, 3, coefImg_mdre_path, 11),
    ]
    
    for ds_name, in_zarr, dates, nharm, out_zarr, iffitted in datasets:
        # Check if the input zarr exists.
        if not os.path.exists(in_zarr):
            print(f"Skipping {ds_name}: input zarr {in_zarr} does not exist")
            continue
        
        # Check if the output zarr exists.
        if os.path.exists(out_zarr):
            print(f"{out_zarr} exists -> skipping {ds_name}")
            continue
        
        # Calling functions
        run_for_dataset(
            dataset_zarr_path=in_zarr,
            dataset_name=ds_name,
            dataset_dates=dates,
            out_zarr_path=out_zarr,
            nharm=nharm,
            iffitted=iffitted,
            overwrite_output=True
        )

    print("All datasets processed.")


## 📋 Complete Workflow
'''
Step 1: Run the refactored code

↓
Generates: mdre_lists.zarr and ltre_lists.zarr

Step 2: Run the modified Document 5 code (HANTS processing)

↓
Generates: coefImg_mdre_.zarr and coefImg_ltre_.zarr
'''

Spatial chunking H: (30,) W: (38,)
Dask client: <Client: 'tcp://127.0.0.1:53229' processes=4 threads=4, memory=22.35 GiB>
Submitted 1 tasks for dataset ltre_lists.
Wrote 1/1 blocks for ltre_lists (last block h[0:30] w[0:38])
Finished dataset ltre_lists. Output at: E:\data\HANTS\python_exp\test\coefImg_ltre_.zarr
Spatial chunking H: (30,) W: (38,)
Dask client: <Client: 'tcp://127.0.0.1:53267' processes=4 threads=4, memory=22.35 GiB>
Submitted 1 tasks for dataset mdre_lists.
Wrote 1/1 blocks for mdre_lists (last block h[0:30] w[0:38])
Finished dataset mdre_lists. Output at: E:\data\HANTS\python_exp\test\coefImg_mdre_.zarr
All datasets processed.


'\nStep 1: Run the refactored code\n\n↓\nGenerates: mdre_lists.zarr and ltre_lists.zarr\n\nStep 2: Run the modified Document 5 code (HANTS processing)\n\n↓\nGenerates: coefImg_mdre_.zarr and coefImg_ltre_.zarr\n'

## Calculate the sparsity coefficient for the predicted year (rewrite repeatedly)

In [8]:
# ------------------ Run pipeline for all datasets ------------------
if __name__ == "__main__":
    # Ensure ltpr_lists and mdpr_lists are already in ZARR format
    # Use the ZARR path generated by your previous data preparation code
    
    datasets = [
        # (Dataset name, zarr path, date array, harmonic number, output zarr path, iffitted parameter)
        ('ltpr_lists', ltpr_lists_path, ltpr_dates, 1, coefImg_ltpr_path, 2),
        ('mdpr_lists', mdpr_lists_path, mdpr_dates, 3, coefImg_mdpr_path, 11)
    ]

    for ds_name, in_zarr, dates, nharm, out_zarr, iffitted in datasets:
        if not os.path.exists(in_zarr):
            print(f"Skipping {ds_name}: zarr file {in_zarr} does not exist")
            continue
        
        # Calling function
        run_for_dataset(
            dataset_zarr_path=in_zarr,
            dataset_name=ds_name,
            dataset_dates=dates,
            out_zarr_path=out_zarr,
            nharm=nharm,
            iffitted=iffitted,
            overwrite_output=True
        )

    print("All datasets processed.")

Spatial chunking H: (30,) W: (38,)
Dask client: <Client: 'tcp://127.0.0.1:53308' processes=4 threads=4, memory=22.35 GiB>
Submitted 1 tasks for dataset ltpr_lists.
Wrote 1/1 blocks for ltpr_lists (last block h[0:30] w[0:38])
Finished dataset ltpr_lists. Output at: E:\data\HANTS\python_exp\test\coefImg_ltpr_.zarr
Spatial chunking H: (30,) W: (38,)
Dask client: <Client: 'tcp://127.0.0.1:53349' processes=4 threads=4, memory=22.35 GiB>
Submitted 1 tasks for dataset mdpr_lists.
Wrote 1/1 blocks for mdpr_lists (last block h[0:30] w[0:38])
Finished dataset mdpr_lists. Output at: E:\data\HANTS\python_exp\test\coefImg_mdpr_.zarr
All datasets processed.


## Calculate weight

In [9]:
import os, numpy as np, zarr
from datetime import datetime as dt_class

from dask.distributed import Client, LocalCluster, as_completed
import numcodecs, shutil

# ---------------- User Parameter ----------------
alpha = 2.0
H_chunk = 64
W_chunk = 64
n_workers = 4
threads_per_worker = 1
memory_limit_per_worker = '6GB'

# -------------------------------------------

def dates_to_ts(dates):
    return np.array([dt_class.combine(d, dt_class.min.time()).timestamp() for d in dates], dtype=np.float64)

# Function:calculate weight
def worker_compute_weight(h0, h1, w0, w1, ltpr_lists_path_local, ltpr_dates_ts_local, alpha_local):
    import zarr as _zarr, numpy as _np
    zr = _zarr.open(ltpr_lists_path_local, mode='r')
    block = zr['ltpr_lists'][:, h0:h1, w0:w1]  # (T, bh, bw)
    block = _np.asarray(block, dtype=_np.float32)

    T, bh, bw = block.shape
    out = _np.full((bh, bw), _np.nan, dtype=_np.float32)

    for ii in range(bh):
        for jj in range(bw):
            series = block[:, ii, jj]
            valid_mask = ~_np.isnan(series)
            dates_sel = ltpr_dates_ts_local[valid_mask]
            len1 = dates_sel.size
            if len1 >= 10:
                out[ii, jj] = 1.0
            elif len1 <= 3:
                out[ii, jj] = 0.0
            else:
                gaps = _np.diff(dates_sel) / 86400.0  # second becomes day
                if gaps.size == 0:
                    out[ii, jj] = 0.0
                else:
                    mean_gap = _np.nanmean(gaps)
                    std_gap  = _np.nanstd(gaps)
                    CV = 0.0 if mean_gap == 0 else (std_gap / mean_gap + 1e-6)
                    U = _np.exp(-alpha_local * CV)
                    Q = _np.tanh(len1 / 3.0) #Quantity Factors
                    out[ii, jj] = float(min(U * Q, 1))  
    return out

def run_parallel_weight_existing():
    zr = zarr.open(ltpr_lists_path, mode='r')
    arr = zr['ltpr_lists']
    T, H, W = arr.shape
    print("Loaded ltpr zarr:", ltpr_lists_path, "shape:", arr.shape, "chunks:", arr.chunks)

    if os.path.exists(weight_zarr_path):
        shutil.rmtree(weight_zarr_path)

    store_out = zarr.DirectoryStore(weight_zarr_path)
    root_out = zarr.group(store=store_out, overwrite=True)
    compressor = numcodecs.Blosc(cname='zstd', clevel=3, shuffle=numcodecs.Blosc.SHUFFLE)
    out_ds = root_out.create_dataset('weight', shape=(H, W), chunks=(H_chunk, W_chunk), dtype='f4', compressor=compressor)
    print("Created output weight zarr:", weight_zarr_path, "shape:", out_ds.shape)

    # Get block distribution
    def norm_chunk(c, full):
        if isinstance(c, int):
            size = c
            parts = [size]*(full//size)
            rem = full%size
            if rem: parts.append(rem)
            return parts
        else:
            return list(c)

    h_chunks = norm_chunk(arr.chunks[1], H)
    w_chunks = norm_chunk(arr.chunks[2], W)
    print("Spatial chunks:", h_chunks, w_chunks)

    # Start Dask
    cluster = LocalCluster(n_workers=n_workers, threads_per_worker=threads_per_worker, memory_limit=memory_limit_per_worker)
    client = Client(cluster)
    print("Dask cluster started.")

    # Convert date to timestamp
    dates_ts = dates_to_ts(ltpr_dates)

    futures = {}
    for hi, hsize in enumerate(h_chunks):
        h0 = sum(h_chunks[:hi])
        h1 = h0 + hsize
        for wi, wsize in enumerate(w_chunks):
            w0 = sum(w_chunks[:wi])
            w1 = w0 + wsize
            f = client.submit(worker_compute_weight, h0, h1, w0, w1, ltpr_lists_path, dates_ts, alpha, pure=False)
            futures[f] = (h0, h1, w0, w1)

    total = len(futures)
    print(f"Submitted {total} tasks")

    done = 0
    for f in as_completed(futures):
        try:
            res = f.result()
        except Exception as e:
            print("Worker error:", e)
            continue
        h0, h1, w0, w1 = futures[f]
        out_ds[h0:h1, w0:w1] = res
        done += 1
        if done % 10 == 0 or done == total:
            print(f"Wrote {done}/{total} blocks")

    client.close()
    cluster.close()
    print("✅ Weight map done. Saved to:", weight_zarr_path)

if __name__ == "__main__":
    run_parallel_weight_existing()

Loaded ltpr zarr: E:\data\HANTS\python_exp\test\ltpr_lists.zarr shape: (36, 30, 38) chunks: (36, 64, 64)
Created output weight zarr: E:\data\HANTS\python_exp\test\weight_.zarr shape: (30, 38)
Spatial chunks: [30] [38]
Dask cluster started.
Submitted 1 tasks
Wrote 1/1 blocks
✅ Weight map done. Saved to: E:\data\HANTS\python_exp\test\weight_.zarr


# Significance removal

## Landsat reference sequence significance

In [10]:
import zarr 
import dask.array as da
import gc
 
# Initialize the Zarr array (lazy loading)
coefImg_ltre = zarr.open(coefImg_ltre_path, mode='r')['coefs']
coef_dask = da.from_zarr(coefImg_ltre)
 
# Calculate the energy term (inert).
energy_zero = coef_dask[0,:,:]**2
energy_year = coef_dask[1,:,:]**2
energy_half = coef_dask[3,:,:]**2 
energy_quarter = coef_dask[5,:,:]**2
total_energy = energy_zero + energy_year + energy_half + energy_quarter
 
# Energy percentage (inertia)
ratio01 = (energy_zero + energy_year) / total_energy 
ratio012 = (energy_zero + energy_year + energy_half) / total_energy
 
# Mask generation (inert)
# Execution rules:
threshold = 0.97
mask2 = da.ones_like(energy_half, dtype=bool)
mask3 = da.ones_like(energy_quarter, dtype=bool)
# Scenario 1: Zero frequency + annual cycle energy percentage > 97%, but zero frequency < 97%, shield A2, A3
mask2 = da.where(ratio01 > threshold, False, mask2)
mask3 = da.where(ratio01 > threshold, False, mask3)
# Scenario 2: Zero + Year + Half a Year > 95%, but Zero + Year < 95%, A3 blocked
mask3 = da.where((ratio01 <= threshold) & (ratio012 > threshold), False, mask3)

del total_energy,energy_zero,energy_year,energy_half,energy_quarter
gc.collect()

# Apply a mask (inert).
amp2 = da.where(mask2, coef_dask[3,:,:], np.nan)
amp3 = da.where(mask3, coef_dask[5,:,:], np.nan)
pha2 = da.where(mask2, coef_dask[4,:,:], np.nan)
pha3 = da.where(mask3, coef_dask[6,:,:], np.nan)
 
# Stacked results (lazy)
coefImg_ltre = da.stack([
    coef_dask[0], coef_dask[1], 
    amp2, amp3, 
    coef_dask[2], 
    pha2, pha3 
], axis=0)

del amp2, amp3,  pha2, pha3, coef_dask
gc.collect()

# Option：Save to Zarr
#coefImg_ltre.to_zarr('output.zarr', overwrite=True)

0

## MODIS reference sequence significance

In [12]:
import gc

coefImg_mdre = zarr.open(coefImg_mdre_path, mode='r')['coefs'] 
coef_dask = da.from_zarr(coefImg_mdre)

energy_zero = coef_dask[0,:,:]**2
energy_year = coef_dask[1,:,:]**2
energy_half = coef_dask[3,:,:]**2 
energy_quarter = coef_dask[5,:,:]**2
total_energy   = energy_zero + energy_year + energy_half + energy_quarter

ratio01  = (energy_zero + energy_year)                / total_energy
ratio012 = (energy_zero + energy_year + energy_half)  / total_energy

mask2 = da.ones_like(energy_half, dtype=bool)
mask3 = da.ones_like(energy_quarter, dtype=bool)
del total_energy,energy_zero,energy_year,energy_half,energy_quarter
gc.collect()


threshold = 0.97
# Scenario 1: Zero frequency + annual cycle energy percentage > 97%, but zero frequency < 97%, shield A2, A3
mask2 = da.where(ratio01 > threshold, False, mask2)
mask3 = da.where(ratio01 > threshold, False, mask3)
# Scenario 2: Zero + Year + Half a Year > 95%, but Zero + Year < 95%, A3 blocked
mask3 = da.where((ratio01 <= threshold) & (ratio012 > threshold), False, mask3)

amp2 = da.where(mask2, coef_dask[3,:,:], np.nan)
amp3 = da.where(mask3, coef_dask[5,:,:], np.nan)
pha2 = da.where(mask2, coef_dask[4,:,:], np.nan)
pha3 = da.where(mask3, coef_dask[6,:,:], np.nan)

#####################################################################################################################################################################################
coefImg_mdre = da.stack([
    coef_dask[0], coef_dask[1], 
    amp2, amp3, 
    coef_dask[2], 
    pha2, pha3 
], axis=0)

del amp2, amp3,  pha2, pha3, coef_dask
gc.collect()

0

# Constructing the predicted annual harmonic coefficient

In [13]:
import zarr
import dask.array as da 
import numpy as np
 
# === 配置层（解耦硬编码，提升可维护性） ===
CHUNK_SIZE = (1, 512, 512)  # 推荐：(C, H, W) 适配遥感影像分块；根据实际内存调整
DTYPE = np.float32          
 
# === 一、构建惰性谐波系数张量（全部为 dask.array，底层绑定 zarr） ===
# 启用 chunk-aware 计算
coefImg_mdpr = da.from_zarr(coefImg_mdpr_path, component='coefs', storage_options={})
 
# 【0-1阶振幅】amp_01 = mdpr[0:2] / (mdre[0:2] + ε) * ltre[0:2]
amp_01 = (coefImg_mdpr[0:2] / (coefImg_mdre[0:2] + 1e-6) * coefImg_ltre[0:2]).astype(DTYPE)
 
# 【一阶相位】pha_1 = mdpr[2] - mdre[4] + ltre[4] → 归一化至 [-180, 180)
def normalize_phase(x):
    return ((x + 180) % 360) - 180 
pha_1 = da.map_blocks(normalize_phase,coefImg_mdpr[2:3] - coefImg_mdre[4:5] + coefImg_ltre[4:5]
                      ,dtype=DTYPE)
 
# 【二阶振幅】amp_2 = mdpr[3] / mdre[2] * ltre[2]，并 mask amp_2 >= 1 → 设为 NaN
amp_2_raw = coefImg_mdpr[3:4] / coefImg_mdre[2:3] * coefImg_ltre[2:3]
amp_2 = da.where(amp_2_raw < 1, amp_2_raw, np.nan).astype(DTYPE)
 
# 【二阶相位】pha_2 = mdpr[4] - mdre[5] + ltre[5]，仅在 amp_2 有效处保留
pha_2_raw = coefImg_mdpr[4:5] - coefImg_mdre[5:6] + coefImg_ltre[5:6]
pha_2 = da.map_blocks(normalize_phase,pha_2_raw,dtype=DTYPE)
pha_2 = da.where(da.isfinite(amp_2), pha_2, np.nan)  
 
# 【三阶振幅 & 相位】
amp_3_raw = coefImg_mdpr[5:6] / coefImg_mdre[3:4] * coefImg_ltre[3:4]
amp_3 = da.where(amp_3_raw < 1, amp_3_raw, np.nan).astype(DTYPE)
 
pha_3_raw = coefImg_mdpr[6:7] - coefImg_mdre[6:7] + coefImg_ltre[6:7]
pha_3 = da.map_blocks(normalize_phase, pha_3_raw, dtype=DTYPE)
pha_3 = da.where(da.isfinite(amp_3), pha_3, np.nan)
 
# === 二、加权融合低频策略（Zarr-native weighted harmonic synthesis）===
weight = da.from_zarr(weight_zarr_path, component='weight', storage_options={})
coefImg_ltpr = da.from_zarr(coefImg_ltpr_path, component='coefs', storage_options={})
 
def weighted_harmonic_synthesis_dask(amp1, amp2, pha1, pha2, weight):
    c1 = amp1 * da.exp(1j * da.deg2rad(pha1))
    c2 = amp2 * da.exp(1j * da.deg2rad(pha2))
    result_complex = da.where(weight==0, 0, weight * c1) + (1-weight)*c2    
    result_amp = da.abs(result_complex)
    result_pha_rad = da.angle(result_complex)
    result_pha_deg = da.rad2deg(result_pha_rad)
    result_pha_deg = ((result_pha_deg + 180) % 360) - 180
    return result_amp, result_pha_deg
 
amp_0_w = da.where(weight == 0, 0, weight * coefImg_ltpr[0:1]) + (1 - weight) * amp_01[0:1]
amp_1_w, pha_1_w = weighted_harmonic_synthesis_dask(coefImg_ltpr[1:2],amp_01[1:2],coefImg_ltpr[2:3],pha_1,weight=weight)
 
# === 三、构建最终融合系数张量（沿 channel axis=0 拼接）===
# 注意：所有输入必须具有相同 chunk shape！强制 rechunk 保障兼容性

def safe_rechunk(arr, chunks=CHUNK_SIZE):
    if arr.chunks != chunks:
        return arr.rechunk(chunks)
    return arr
 
coefImg_fusn = da.concatenate([
    safe_rechunk(amp_0_w,      CHUNK_SIZE),
    safe_rechunk(amp_1_w,      CHUNK_SIZE),
    safe_rechunk(amp_2,        CHUNK_SIZE),
    safe_rechunk(amp_3,        CHUNK_SIZE),
    safe_rechunk(pha_1_w,      CHUNK_SIZE),
    safe_rechunk(pha_2,        CHUNK_SIZE),
    safe_rechunk(pha_3,        CHUNK_SIZE),
], axis=0)
 
'''Option:
# === 四、 写入 Zarr===
output_store = zarr.DirectoryStore(coefImg_fusn_path)
output_root = zarr.group(store=output_store, overwrite=True)
 
# 创建目标数组（预分配，提升 I/O 效率）
target_shape = coefImg_fusn.shape
target_chunks = (1,) + CHUNK_SIZE[1:]  # (7, H, W) → 每个 band 独立 chunk
fusn_zarr = output_root.create_dataset(
    'coefs',
    shape=target_shape,
    chunks=target_chunks,
    dtype=DTYPE,
    fill_value=np.nan,  # 显式声明 NaN 填充值，提升压缩率
    compressor=zarr.Blosc(cname='zstd', clevel=3, shuffle=zarr.Blosc.SHUFFLE)  # 推荐压缩器 
)
 
# 🔥 触发全量计算并流式写入（内存可控！）
print(f"→ 开始持久化融合系数：{target_shape}，chunk={target_chunks}...")
coefImg_fusn.store(fusn_zarr, lock=False, compute=True)  # dask 自动调度分块写入
 
print("✅ 融合系数已以 Zarr 原生格式成功写入！")
print(f"   • 存储路径：{coefImg_fusn_path}")
print(f"   • 总大小：{fusn_zarr.nbytes_stored / 1024**3:.2f} GB (压缩后)")
'''

'Option:\n# === 四、 写入 Zarr===\noutput_store = zarr.DirectoryStore(coefImg_fusn_path)\noutput_root = zarr.group(store=output_store, overwrite=True)\n \n# 创建目标数组（预分配，提升 I/O 效率）\ntarget_shape = coefImg_fusn.shape\ntarget_chunks = (1,) + CHUNK_SIZE[1:]  # (7, H, W) → 每个 band 独立 chunk\nfusn_zarr = output_root.create_dataset(\n    \'coefs\',\n    shape=target_shape,\n    chunks=target_chunks,\n    dtype=DTYPE,\n    fill_value=np.nan,  # 显式声明 NaN 填充值，提升压缩率\n    compressor=zarr.Blosc(cname=\'zstd\', clevel=3, shuffle=zarr.Blosc.SHUFFLE)  # 推荐压缩器 \n)\n \n# 🔥 触发全量计算并流式写入（内存可控！）\nprint(f"→ 开始持久化融合系数：{target_shape}，chunk={target_chunks}...")\ncoefImg_fusn.store(fusn_zarr, lock=False, compute=True)  # dask 自动调度分块写入\n \nprint("✅ 融合系数已以 Zarr 原生格式成功写入！")\nprint(f"   • 存储路径：{coefImg_fusn_path}")\nprint(f"   • 总大小：{fusn_zarr.nbytes_stored / 1024**3:.2f} GB (压缩后)")\n'

# Harmonic coefficient reconstructed 

## FMSF Reconstruction

In [14]:
import zarr 
import dask.array as da 
from datetime import datetime
import pandas as pd
 
def harmonic_reconstructed(coefImg_fusion, dates, num_harm):     
    if num_harm == 1: 
        constant    = coefImg_fusion[0] 
        amplitude_1 = coefImg_fusion[1] 
        phase_1     = coefImg_fusion[2] * np.pi / 180 
        cos_1 = da.cos(phase_1) * amplitude_1 
        sin_1 = da.sin(phase_1) * amplitude_1 
        harmonicTrendCoefficients_reconstructed = da.stack([constant, cos_1, sin_1], axis=0) 
    else: 
        constant = coefImg_fusion[0] 
        amplitude_1, amplitude_2, amplitude_3 = coefImg_fusion[1], coefImg_fusion[2], coefImg_fusion[3] 
        phase_1, phase_2, phase_3 = coefImg_fusion[4]*np.pi/180, coefImg_fusion[5]*np.pi/180, coefImg_fusion[6]*np.pi/180 
        cos_1, sin_1 = da.cos(phase_1)*amplitude_1, da.sin(phase_1)*amplitude_1 
        cos_2, sin_2 = da.cos(phase_2)*amplitude_2, da.sin(phase_2)*amplitude_2 
        cos_3, sin_3 = da.cos(phase_3)*amplitude_3, da.sin(phase_3)*amplitude_3 
        harmonicTrendCoefficients_reconstructed = da.stack([constant, cos_1, sin_1, cos_2, sin_2, cos_3, sin_3], axis=0) 
    
    # 2. Constructing time basis functions
    base_date = datetime(2020, 1, 1).date() 
    days = da.array([(d - base_date).days for d in dates]) 
    t = days / 365.25 * 2 * np.pi 
    
    # 3. Generate Fourier basis matrix 
    components = [da.ones_like(t)]  # constant term，创建一个和t 形状相同的数组，所有值都是 1
    for i in range(1, num_harm+1): 
        components.extend([da.cos(i*t), da.sin(i*t)]) 
        
    X = da.stack(components, axis=1)  # (n, 2*num_harm+1) 

    
    # 4. Broadcasting and product operations
    X_final = da.broadcast_to(X[:, :, None, None],
                              (X.shape[0], harmonicTrendCoefficients_reconstructed.shape[0], coefImg_fusion.shape[1], coefImg_fusion.shape[2])) 
    y_pred = da.nansum(harmonicTrendCoefficients_reconstructed * X_final, axis=1)  # 沿系数轴求和 

    return y_pred 

## HANTS Reconstruction

In [15]:
def hants_reconstructed(coefImg, dates, num_harm):
    constant = coefImg[0]    
    # 1. Extract all amplitudes and phases
    amplitudes = coefImg[1:num_harm+1]
    phases = coefImg[num_harm+1:2*num_harm+1] * np.pi / 180  # 转换为弧度

    # 2. Constructing time basis functions
    base_date = datetime(2020, 1, 1).date() 
    days = da.array([(d - base_date).days for d in dates]) 
    t = days / 365.25 * 2 * np.pi 
    
    # Initialization result list
    coefficients = [constant]
    components = [da.ones_like(t)]  # constant term，创建一个和t 形状相同的数组，所有值都是 1
        
    # Loop through the cos and sin terms for each harmonic
    for i in range(num_harm):
        cos_i = da.cos(phases[i]) * amplitudes[i]
        sin_i = da.sin(phases[i]) * amplitudes[i]
        coefficients.extend([cos_i, sin_i])
        components.extend([da.cos((i+1)*t), da.sin((i+1)*t)]) 
    
    # Stack all coefficients
    harmonicTrendCoefficients_reconstructed = da.stack(coefficients, axis=0)       
    X = da.stack(components, axis=1)  # (n, 2*num_harm+1) 
    
    # 4. Broadcasting and product operations
    X_final = da.broadcast_to(X[:, :, None, None],
                              (X.shape[0], harmonicTrendCoefficients_reconstructed.shape[0], coefImg.shape[1], coefImg.shape[2])) 
    y_pred = da.nansum(harmonicTrendCoefficients_reconstructed * X_final, axis=1)  # 沿系数轴求和 
    return y_pred 

#Hants harmonic coefficients of the 3rd harmonic
if __name__ == "__main__":
    
    datasets = [
        # (Dataset, zarr path, date , harmonic , output zarr , iffitted)
        ('ltpr_lists', hants_lists_path, ltpr_dates, 3, coefImg_hants_path, 11)
    ]

    for ds_name, in_zarr, dates, nharm, out_zarr, iffitted in datasets:
        if not os.path.exists(in_zarr):
            print(f"Skipping {ds_name}: zarr file {in_zarr} does not exist")
            continue
        
        # Calling function
        run_for_dataset(
            dataset_zarr_path=in_zarr,
            dataset_name=ds_name,
            dataset_dates=dates,
            out_zarr_path=out_zarr,
            nharm=nharm,
            iffitted=iffitted,
            overwrite_output=True
        )

Spatial chunking H: (30,) W: (38,)
Dask client: <Client: 'tcp://127.0.0.1:53470' processes=4 threads=4, memory=22.35 GiB>
Submitted 1 tasks for dataset ltpr_lists.
Wrote 1/1 blocks for ltpr_lists (last block h[0:30] w[0:38])
Finished dataset ltpr_lists. Output at: E:\data\HANTS\python_exp\test\coefImg_hants_.zarr


## Combined the results of FMSF and HANTS

In [16]:
# Calculate a continuous time series
from datetime import date, timedelta 
  
start_date = date(year, 1, 1)   
end_date   = date(year + 1, 1, 1)  
date_series = pd.date_range(start=start_date, end=end_date, freq=f'{interval}D').date


coefImg_lazy = da.from_zarr(coefImg_hants_path, component='coefs')
new_order = [0, 1, 3, 5, 2, 4, 6]
coefImg_hants = coefImg_lazy[new_order, :, :]

ltpr_lists = zarr.open(ltpr_lists_path, mode='r')['ltpr_lists'][:,:,:] 
count_mask = da.sum(~da.isnan(ltpr_lists), axis=0) > 7 

coefImg_hants_ = da.where(count_mask, coefImg_hants, np.nan)
coefImg_fusn_  = da.where(~count_mask, coefImg_fusn , np.nan)

#date_series = ltpr_dates

rendvi_hants_masked = hants_reconstructed(coefImg_hants_, date_series, 3)
rendvi_fusn_masked  = harmonic_reconstructed(coefImg_fusn_, date_series, 3)

rendvi_combined = np.zeros((len(date_series), count_mask.size))
rendvi_combined = da.where(count_mask, rendvi_hants_masked, rendvi_fusn_masked)

#del coefImg_fusn,coefImg_hants
#gc.collect()

# Export the reconstructed image

In [19]:
import rasterio
from rasterio.transform import from_origin

export_tif    = rendvi_combined

height, width = export_tif.shape[1:]
count         = export_tif.shape[0]
band_names = [d.strftime("%Y-%m-%d") for d in date_series] 

os.makedirs(output_dir, exist_ok=True)

profile = {
    "driver": "GTiff",
    "dtype": "float32",
    "count": count,
    "height": height,
    "width": width,
    "crs": "EPSG:4326",
    "transform": transform,
    "nodata": np.nan
}

filename = os.path.join(output_dir, f"rendvi_{year}.tif")
os.makedirs(output_dir, exist_ok=True)

with rasterio.open(filename, "w", **profile) as dst:
    for i in range(count):
        band_idx = i + 1       
        dst.write(export_tif[i].astype(np.float32), band_idx)
        dst.set_band_description(band_idx, band_names[i])

print(f"Export Finished")

Export Finished
